# CS 297 - Experiment 1
### Description: Can a small, open weight model reliably detect noise in an image and determine whether denoising is required?
### Ethan Machado

### Original configuration with few-shot references and prediction logging

A fresh full run using the recovered model, image processing and generation configuration, extended with four labeled visual examples from held-out subject **0**. The reference subject is excluded from evaluation: **196 target images, 49 per severity**. Historical scores and offline historical review cells are omitted.

Preserved: **Qwen/Qwen3.5-0.8B**, original base prompt, one target at a time, one pass, greedy generation, **128 maximum new tokens**, RGB conversion, and the processor's default resizing/alignment. No custom pixel cap, crop, EXIF rotation or dtype override is added.

Few-shot change: each conversation contains the held-out subject's clean/weak/medium/strong images with their correct JSON answers, followed by the target. The shared runner appends its few-shot calibration guidance to the original base prompt. Five whole images per prediction require more memory and time than the zero-shot notebook.

Includes the same progress bar, per-image raw responses/errors/checkpoints, run metadata, CSV exports and post-run evaluation. Use the project `.venv` kernel, then **Restart Kernel → Run All** to start inference. Cached model files are used. Results are saved separately from zero-shot runs.

## Setup and original base prompt

The recovered base prompt is embedded below. The existing few-shot guidance is appended by the shared runner and the complete effective prompt is saved in the manifest. Normal processor resizing remains enabled; “before resize cap” does not mean `do_resize=False`.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_PATH = Path.cwd().resolve()
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    PROJECT_PATH = PROJECT_PATH.parent
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    raise RuntimeError("Start the notebook from experiment1 or its notebooks directory")
sys.path.insert(0, str(PROJECT_PATH))
os.environ["HF_HOME"] = str(PROJECT_PATH / "hf_cache")

import json
import uuid
from datetime import datetime, timezone
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from transformers import AutoProcessor, AutoModelForMultimodalLM
from src.image_quality_pipeline import build_dataset, run_inference, load_results, prepare_few_shot_split, FEW_SHOT_GUIDANCE
from src.evaluation import evaluate_saved_run

MODEL_NAME = "Qwen/Qwen3.5-0.8B"
EXAMPLE_SUBJECT = "0"  # same reference subject as the existing few-shot setup
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = PROJECT_PATH / "outputs" / "original_configuration_few_shot_runs" / SESSION_ID
RUN_DIR = OUTPUT_DIR / "experiment_001" / "run_001"
print("New run directory:", RUN_DIR)


In [ ]:
ORIGINAL_PROMPT = 'You are performing a controlled image-quality classification task.\n\nInspect only the supplied face image. Do not describe the person, identity, facial expression, or content. Estimate whether visible image degradation or noise is present.\n\nUse these labels exactly:\n\nnoise:\n- "0" = no obvious image noise or degradation\n- "1" = visible image noise or degradation\n\nseverity:\n- "0" = clean; no obvious degradation\n- "1" = weak; slight degradation, but facial details remain clear\n- "2" = medium; obvious degradation that affects visible facial details\n- "3" = strong; severe degradation that substantially obscures facial details\n\nRouting policy:\n- Return route "0" for severity "0" or "1" (skip denoising).\n- Return route "1" for severity "2" or "3" (apply denoising).\n\nThe route must follow the predicted severity.\n\nReturn exactly one valid JSON object.\nDo not include Markdown, code fences, explanations, or additional keys.\n\nRequired format:\n{"noise":"0" or "1","severity":"0","1","2","3","route":"0" or "1"}\n\nExample expected output:\n{"noise":"1","severity":"2","route":"1"}\n'

print("Effective few-shot prompt (guidance is appended once by the runner):")
print(ORIGINAL_PROMPT + FEW_SHOT_GUIDANCE)


## Evaluation dataset and held-out visual references

Subject `EXAMPLE_SUBJECT` supplies one labeled reference for each severity. All four images of this subject are excluded from targets and evaluation. The remaining 49 subjects provide 196 targets, retaining the original severity-group and filename ordering. The table below shows exactly which references will be supplied. Select the reference subject before examining predictions.

The zero-shot sibling evaluates 200 images. For a matched comparison, evaluate its saved predictions on these same 196 targets.

In [ ]:
df_experiment = build_dataset(PROJECT_PATH / "data/files")
if len(df_experiment) != 200 or df_experiment.groupby("severity").size().to_dict() != {"0": 50, "1": 50, "2": 50, "3": 50}:
    raise ValueError("Expected the original balanced dataset: 200 images, 50 per severity")
subject_counts = df_experiment.groupby(["subject_id", "severity"]).size().unstack(fill_value=0)
if len(subject_counts) != 50 or not subject_counts.eq(1).all().all():
    raise ValueError("Expected 50 subjects with exactly one image per severity")
print("Total images:", len(df_experiment))
display(df_experiment.groupby(["augmentation", "severity"], sort=False).size().rename("images").to_frame())
display(df_experiment.head())

df_evaluation, few_shot_examples = prepare_few_shot_split(df_experiment, EXAMPLE_SUBJECT)
assert len(df_evaluation) == 196
assert df_evaluation.groupby("severity").size().to_dict() == {"0": 49, "1": 49, "2": 49, "3": 49}
assert EXAMPLE_SUBJECT not in set(df_evaluation["subject_id"].astype(str))
assert not set(df_evaluation["file_path"]) & {example.image_path for example in few_shot_examples}
print("Reference subject:", EXAMPLE_SUBJECT, "| Evaluation targets:", len(df_evaluation))
display(pd.DataFrame([{"image_path": example.image_path, "subject_id": example.subject_id,
                       **example.prediction} for example in few_shot_examples]))
display(df_evaluation.groupby(["augmentation", "severity"], sort=False).size().rename("evaluation_images").to_frame())


## Load model and processor

As in the recovered run, the device is MPS when available, otherwise CPU, and no dtype is explicitly requested. Defaults are checked rather than overwritten. The original output showed `mps:0`; inspect the device below before starting inference if comparing runtimes.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, local_files_only=True)
processor_size = dict(processor.image_processor.size)
if processor_size.get("longest_edge") != 16777216 or processor_size.get("shortest_edge") != 65536:
    raise ValueError(f"Cached processor defaults differ from the recovered configuration: {processor_size}")
if not processor.image_processor.do_resize:
    raise ValueError("The recovered processor used normal resizing")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForMultimodalLM.from_pretrained(MODEL_NAME, local_files_only=True).to(device)
print("Device:", model.device)
print("Actual dtype:", model.dtype)
print("Processor size:", processor_size)


## Few-shot inference adapter

Each independent conversation contains four visual reference/user turns with labeled assistant answers and one target/user turn. The effective prompt is included with the first reference. Reference images and targets use the original RGB loading, processor defaults, chat-template arguments, greedy generation and decoding. The runner calls `generate` sequentially with batch size 1. Images are closed after each prediction; no conversation history carries across targets.

In [ ]:
from contextlib import ExitStack

class OriginalConfigurationFewShotModel:
    supports_few_shot = True

    def generate(self, image_path, prompt, *, examples=()):
        if len(examples) != 4:
            raise ValueError("Expected four labeled visual reference examples")
        with ExitStack() as stack:
            def image_turn(path, text):
                source = stack.enter_context(Image.open(path))
                image = stack.enter_context(source.convert("RGB"))
                return {"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text},
                ]}

            messages = []
            for index, example in enumerate(examples):
                reference_prompt = ((prompt + "\n\n") if index == 0 else "")
                reference_prompt += "Reference image: classify its degradation."
                messages.append(image_turn(example.image_path, reference_prompt))
                messages.append({"role": "assistant", "content": [
                    {"type": "text", "text": json.dumps(example.prediction)},
                ]})
            messages.append(image_turn(image_path, "Target image: return only its JSON classification."))

            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)
            with torch.inference_mode():
                outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
            return processor.decode(generated_tokens, skip_special_tokens=True).strip()

model_adapter = OriginalConfigurationFewShotModel()


In [ ]:
# Metadata only: these attributes do not change adapter behavior.
model_adapter.inference_config = {
    "model_source": MODEL_NAME,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "device": str(model.device), "dtype": str(model.dtype),
    "max_new_tokens": 128, "do_sample": False,
    "processor_size": processor_size,
    "processor_do_resize": processor.image_processor.do_resize,
    "processor_config": processor.to_dict(),
    "image_views": "one_whole_image", "image_conversion": "RGB",
    "exif_transpose": False, "detail_crop_size": None,
    "custom_pixel_cap": None, "passes": 1,
    "reference_subject": EXAMPLE_SUBJECT, "reference_images_per_target": 4,
    "total_images_per_conversation": 5,
    "reference_conversation": "four_labeled_user_assistant_pairs_then_target",
}


## Run and checkpoint each image

The progress bar shows completed images and failures. Each `batch_XXXXX.json` contains one image's raw model output, parsed prediction or error, and elapsed time. `labels.json` stores ground truth keyed by the same image path; `manifest.json` records the exact effective prompt, reference paths/labels, held-out subject, configuration, progress and total runtime. Invalid JSON or inconsistent labels remain failures; responses are not repaired or retried.

The runner saves `metrics.json` after the full pass finishes. If interrupted, completed image checkpoints remain available and the manifest records the interruption. This notebook does not silently rerun or overwrite an existing session: rerun the setup cell to create a new session. The next section can inspect saved checkpoints without loading the model.

In [ ]:
if RUN_DIR.exists():
    raise FileExistsError(f"Run already exists: {RUN_DIR}. Use the offline evaluation section, or rerun setup for a new session.")
results = run_inference(
    model=model_adapter,
    image_paths=df_evaluation["file_path"].tolist(),
    prompt=ORIGINAL_PROMPT,
    batch_size=1,
    output_dir=OUTPUT_DIR,
    experiment_number=1,
    run_number=1,
    show_progress=True,
    labels=df_evaluation,
    model_name=MODEL_NAME,
    inference_mode="few_shot",
    examples=few_shot_examples,
    response_format="json",
)
print("Finished; saved", len(results), "individual results to", RUN_DIR)


## Evaluate and export saved predictions

Run this section after inference finishes. For later offline review, set `RUN_DIR` below to a saved run folder; neither the setup cells nor the model are needed. Incomplete runs can be inspected, but are explicitly identified and are not presented as full-run scores.

Accuracy, precision, recall, macro-F1, confusion matrices and severity MAE use valid predictions. Coverage and failures are reported separately. “Correct / attempted” treats failed attempts as unsuccessful predictions; “Correct / planned” also exposes images not attempted in an interrupted run. Binary precision/recall use class 1; severity precision/recall use macro averaging. Confusion matrices have true labels in rows and predicted labels in columns.

Only the 196 target images are scored. The four labeled references are excluded from predictions, saved ground truth, metric denominators and CSV exports; their identities and labels are recorded in the manifest.

In [ ]:
# Offline review: replace this value with Path("/absolute/path/to/experiment_001/run_001").
# If RUN_DIR is already defined by the run above, leave this cell unchanged.
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate shared evaluation code without importing/loading a model.
review_root = Path.cwd().resolve()
if not (review_root / "src/evaluation.py").exists():
    review_root = review_root.parent
if str(review_root) not in sys.path:
    sys.path.insert(0, str(review_root))
from src.image_quality_pipeline import load_results
from src.evaluation import evaluate_saved_run

if "RUN_DIR" not in globals():
    raise RuntimeError("Set RUN_DIR to the saved run folder before offline evaluation")
RUN_DIR = Path(RUN_DIR)
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
labels = json.loads((RUN_DIR / "labels.json").read_text())
saved_results = load_results(RUN_DIR)
metrics = evaluate_saved_run(RUN_DIR, save=True)

prediction_rows = []
for batch_path in sorted(RUN_DIR.glob("batch_*.json")):
    batch = json.loads(batch_path.read_text())
    for record in batch["results"]:
        prediction = record["prediction"] or {}
        truth = labels[record["image_path"]]
        row = {"image_path": record["image_path"], "file_name": Path(record["image_path"]).name,
               "subject_id": Path(record["image_path"]).stem,
               "raw_output": record["raw_output"], "error": record["error"],
               "valid": record["prediction"] is not None and record["error"] is None,
               "elapsed_seconds": batch["elapsed_seconds"]}
        for task in ("noise", "severity", "route"):
            row[f"true_{task}"] = truth[task]
            row[f"predicted_{task}"] = prediction.get(task)
            row[f"{task}_correct"] = row["valid"] and prediction.get(task) == truth[task]
        prediction_rows.append(row)
predictions_df = pd.DataFrame(prediction_rows, columns=[
    "image_path", "file_name", "subject_id", "raw_output", "error", "valid", "elapsed_seconds",
    *[name for task in ("noise", "severity", "route")
      for name in (f"true_{task}", f"predicted_{task}", f"{task}_correct")],
])
predictions_df.to_csv(RUN_DIR / "predictions.csv", index=False)
failures_df = predictions_df.loc[~predictions_df["valid"].astype(bool)].copy()
failures_df.to_csv(RUN_DIR / "failures.csv", index=False)

attempted = len(saved_results)
planned = manifest["total_images"]
full_run = manifest["status"] == "completed" and attempted == planned
print("FULL RUN COMPLETE" if full_run else "INCOMPLETE RUN — partial results only")
display(pd.DataFrame([{
    "Status": manifest["status"], "Planned": planned, "Attempted": attempted,
    "Valid": metrics["severity"]["evaluated"], "Failed": metrics["severity"]["failed"],
    "Unattempted": planned - attempted,
    "Coverage of attempts": metrics["severity"]["coverage"],
    "Coverage of planned": metrics["severity"]["label_coverage"],
    "Runtime seconds": manifest.get("elapsed_seconds"),
}]))

summary_rows = []
for task in ("noise", "severity", "route"):
    values = metrics[task]
    summary_rows.append({
        "Task": task, "Correct": values["correct"], "Evaluated": values["evaluated"],
        "Accuracy (valid)": values["accuracy"], "Precision (valid)": values["precision"],
        "Recall (valid)": values["recall"], "Macro-F1 (valid)": values["macro_f1"],
        "MAE (valid)": values.get("mae"),
        "Correct / attempted": values["correct"] / attempted if attempted else None,
        "Correct / planned": values["correct"] / planned if planned else None,
    })
summary_df = pd.DataFrame(summary_rows).set_index("Task")
summary_df.to_csv(RUN_DIR / "evaluation_summary.csv")
display(summary_df)

for task in ("noise", "severity", "route"):
    values = metrics[task]
    print(task, "confusion matrix (valid predictions)")
    display(pd.DataFrame(values["confusion_matrix"],
                         index=pd.Index(values["labels"], name="True"),
                         columns=pd.Index(values["labels"], name="Predicted")))
    print(task, "per-class metrics (valid predictions)")
    display(pd.DataFrame(values["per_class"]).T)

coverage_by_severity = []
for severity in ("0", "1", "2", "3"):
    planned_class = sum(value["severity"] == severity for value in labels.values())
    group = predictions_df.loc[predictions_df["true_severity"] == severity]
    valid = int(group["valid"].sum())
    coverage_by_severity.append({"True severity": severity, "Planned": planned_class,
                                "Attempted": len(group), "Valid": valid, "Failed": len(group) - valid,
                                "Coverage of planned": valid / planned_class if planned_class else None})
coverage_df = pd.DataFrame(coverage_by_severity).set_index("True severity")
coverage_df.to_csv(RUN_DIR / "coverage_by_severity.csv")
display(coverage_df)
print("Individual predictions (first 20):")
display(predictions_df.head(20))
print("Failed images:", len(failures_df))
display(failures_df[["image_path", "true_severity", "raw_output", "error"]])
print("All JSON checkpoints, metrics, and CSV exports:", RUN_DIR)
